
### 04_train_neural_network

#### Purpose

This notebook trains the Telco Churn neural network using PyTorch.

The model architecture was defined and verified in 03_build_neural_network.

The goal of this notebook is to make the model’s initialized weights and biases actually learn from the training data.

The core training flow is:

``` text 

Training Batch
      ↓
Forward Propagation
      ↓
Raw Logits
      ↓
BCEWithLogitsLoss
      ↓
Loss
      ↓
Backpropagation
      ↓
Gradients
      ↓
Adam Optimizer
      ↓
Update Weights + Biases
      ↓
Next Batch
      ↓
Next Epoch

```


#### Production Design

Reusable training logic will live in: src/training.py

The notebook itself will be responsible for:

``` text

Load configuration
      ↓
Prepare training inputs
      ↓
Create a fresh model
      ↓
Configure loss
      ↓
Configure optimizer
      ↓
Run training epochs
      ↓
Run validation
      ↓
Track training history
      ↓
Inspect whether learning occurred

```

So the responsibility split is:

src/model.py
→ defines neural-network architecture

src/preprocessing.py
→ defines preprocessing behavior

src/training.py
→ reusable training and validation functions

04_train_neural_network
→ orchestrates the training experiment


#### Technologies Used

- Databricks
- Python
- PyTorch
- torch.nn
- torch.optim
- scikit-learn preprocessing outputs

Important PyTorch components in this notebook:

- nn.BCEWithLogitsLoss
- torch.optim.Adam
- model.train()
- model.eval()
- torch.no_grad()
- loss.backward()
- optimizer.zero_grad()
- optimizer.step()


#### Input

This notebook requires prepared training and validation data:

- train_loader
- val_loader

with batches shaped approximately:

- X_batch → [32, 45]
- y_batch → [32, 1]

It also requires the reusable neural-network class: TelcoChurnNN from: src/model.py

The model architecture is: 45 → 32 → 16 → 1

where the actual input size is derived from the prepared feature data.


#### Output

This notebook will produce:

- Trained neural-network model
- Training-loss history
- Validation-loss history
- Updated weights and biases
- Training summary

It will also verify that:

``` text

Initial parameters
      ↓
Training
      ↓
Updated parameters

```

which confirms that learning actually occurred.

The trained model will then be evaluated more fully in: 05_evaluate_model


#### Training Architecture

The complete training architecture is:

``` text

TRAINING DATA

     train_loader
          ↓
     Batch
X_batch + y_batch
          ↓
model.train()
          ↓
Forward Propagation
          ↓
     Logits
          ↓
BCEWithLogitsLoss
          ↓
     Loss
          ↓
loss.backward()
          ↓
     Gradients
          ↓
Adam Optimizer
          ↓
Update Weights/Biases
          ↓
     Next Batch
          ↓
Complete Epoch
          ↓
Validation Phase
          ↓
     val_loader
          ↓
     model.eval()
          ↓
Forward Only
          ↓
Validation Loss
          ↓
Track Both Losses
          ↓
     Next Epoch

```

##### 1. Load Project Configuration

In [0]:
%run ./00_project_config

##### 2. Import Required Components

In [0]:
import copy

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.data import prepare_training_data
from src.model import TelcoChurnNN
from src.training import (
    train_one_epoch,
    validate_one_epoch,
)

At this point:

torch
→ tensors and core PyTorch operations

nn
→ neural-network components

TelcoChurnNN
→ reusable model definition

train_one_epoch
→ reusable training logic

validate_one_epoch
→ reusable validation logic

#### What does train_one_epoch() mean?

Conceptually:

``` text

train_one_epoch()
        ↓
Process every training batch once
        ↓
Forward
        ↓
Loss
        ↓
Backward
        ↓
Optimizer update
        ↓
Return average training loss

```
If you have: 155 training batches

then: train_one_epoch() performs approximately:

- 155 forward passes
- 155 loss calculations
- 155 backward passes
- 155 optimizer updates
That equals: 1 epoch


#### What does validate_one_epoch() mean?

Validation does not learn. It performs:

``` text

Validation Batch
      ↓
Forward Propagation
      ↓
Logits
      ↓
Loss
      ↓
Measure only

```

There is:

- NO loss.backward()
- NO optimizer.step()
- NO weight update

That is why validation uses: model.eval() and with torch.no_grad()


#### Training vs Validation


TRAINING

``` text 

model.train()
     ↓
Forward
     ↓
Loss
     ↓
Backward
     ↓
Gradients
     ↓
Optimizer
     ↓
Update Parameters

```


VALIDATION

``` text

model.eval()
     ↓
torch.no_grad()
     ↓
Forward
     ↓
Loss
     ↓
Measure Performance

```

NO parameter updates

#### 3. Prepare Training Data

In [0]:
(
    train_loader,
    val_loader,
    test_loader,
    input_size,
    preprocessor,
) = prepare_training_data(
    spark=spark,
    gold_table=GOLD_TABLE,
    target_column=TARGET_COLUMN,
    columns_to_drop=COLUMNS_TO_DROP,
    numerical_features=NUMERICAL_FEATURES,
    binary_features=BINARY_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    test_size=TEST_SIZE,
    validation_test_size=VALIDATION_TEST_SIZE,
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
)

##### 4. Verify Prepared Training Data

In [0]:
print(
    "Input size:",
    input_size,
)

print(
    "Training batches:",
    len(train_loader),
)

print(
    "Validation batches:",
    len(val_loader),
)

print(
    "Test batches:",
    len(test_loader),
)

##### 5. Inspect One Training Batch

In [0]:
X_batch, y_batch = next(
    iter(train_loader)
)

print(
    "X_batch:",
    X_batch.shape,
)

print(
    "y_batch:",
    y_batch.shape,
)

#### 6. Create Model for Manual Training-Step Verification

In [0]:
model = TelcoChurnNN(
    input_size=input_size,
    hidden_size_1=HIDDEN_SIZE_1,
    hidden_size_2=HIDDEN_SIZE_2,
)

print(model)

##### 7. Define Loss Function

In [0]:
criterion = nn.BCEWithLogitsLoss()


``` text

BCE
↓
Binary Cross-Entropy

WithLogits
↓
expects raw model logits

Loss
↓
measures prediction error

```

model produces: Raw Logit not probability.

Therefore training uses:

``` text

X_batch
   ↓
model
   ↓
LOGITS
   │
   │      y_batch
   │         ↓
   └──────┬──┘
          ↓
BCEWithLogitsLoss
          ↓
LOSS

```

We do not apply Sigmoid ourselves before this loss.

Correct:

logits = model(X_batch)

loss = criterion(
    logits,
    y_batch,
)

Incorrect with this criterion:

probabilities = torch.sigmoid(
    model(X_batch)
)

loss = criterion(
    probabilities,
    y_batch,
)

because BCEWithLogitsLoss expects logits.


##### 8. Define Adam Optimizer

In [0]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

``` text

Adam
↓
Optimizer algorithm

```

``` text

model.parameters()
↓
2,017 weights + biases that can be updated

```

``` text

LEARNING_RATE
↓
Controls size of parameter updates

```

``` text

Adam
uses
↓
gradients
+
gradient history/statistics
+
learning rate
↓
updates weights and biases

```

prepare_training_data()
→ manages the overall data workflow

build_preprocessor()
→ defines how features are transformed

##### Get one training batch

In [0]:
X_batch, y_batch = next(
    iter(train_loader)
)

print(
    "X_batch shape:",
    X_batch.shape,
)

print(
    "y_batch shape:",
    y_batch.shape,
)

#### 9. Save a Weight Before Training

In [0]:
#Let's capture one weight before Adam changes anything:
weight_before = (
    model.layer1.weight[0, 0]
    .detach()
    .clone()
)

print(
    "Weight before training:",
    weight_before.item(),
)

##### Why [0, 0]?

Remember:

layer1.weight
shape = [32,45]

So:

model.layer1.weight[0]

means:

All 45 weights belonging to neuron 1.

Whereas:

model.layer1.weight[0, 0]

means:

The weight connecting input feature 1 → neuron 1.

We're just tracking one parameter so we can visibly prove that training changes it.

##### Why .detach().clone()?

This is worth understanding.

model.layer1.weight[0, 0]

belongs to PyTorch's computation system.

We want a frozen copy of its current value.

.detach()
→ separate this value from gradient tracking

.clone()
→ make an independent copy

Otherwise, we could accidentally retain a reference to something that changes.

##### 10. Clear Previous Gradients

In [0]:
optimizer.zero_grad()

Remember:

PyTorch normally accumulates gradients.

Suppose the previous batch calculated:

gradient = 0.03

and the next batch calculates:

gradient = 0.02

Without clearing, gradients can accumulate.

For normal batch training we want:

Batch 1

zero_grad()
↓
calculate Batch 1 gradients
↓
update


Batch 2

zero_grad()
↓
calculate Batch 2 gradients
↓
update

So:

optimizer.zero_grad()

means:

Clear the previous gradients before calculating gradients for this batch.

Notice:

It does NOT clear weights.

It clears:

gradients

not:

weights
biases

##### 11. Forward Propagation

In [0]:
logits = model(
    X_batch
)

print(
    "Logits shape:",
    logits.shape,
)

print(
    "First 5 logits:"
)

print(
    logits[:5]
)

#These are still raw logits, not probabilities.

##### 12. Calculate the Loss

In [0]:
loss = criterion(
    logits,
    y_batch,
)

print(
    "Batch loss:",
    loss.item(),
)

##### What Gradients Exist Right Now?

So the model has calculated the loss, but has not yet calculated how each parameter contributed to that loss.

Forward propagation ≠ gradient calculation

##### 13. Verify Gradient Before Backpropagation

In [0]:
print(
    "Gradient before backward:",
    model.layer1.weight.grad,
)

##### 14. Backpropagation

In [0]:
loss.backward()

This is the command that triggers:

``` text

Loss
 ↓
Output Layer
 ↓
Hidden Layer 2
 ↓
Hidden Layer 1
 ↓
Calculate gradients

```

PyTorch's automatic differentiation system uses the computation graph it created during the forward pass.

You don't manually calculate thousands of derivatives.

##### 15. Inspect the Gradients

In [0]:
print(
    "Layer 1 gradient shape:",
    model.layer1.weight.grad.shape,
)

##### 16. Inspect One Gradient

In [0]:
gradient = (
    model.layer1.weight.grad[0, 0]
)

print(
    "Gradient for tracked weight:",
    gradient.item(),
)

##### 17. Verify Weight Has Not Changed Yet

No.

We have done:

``` text 

Forward
↓
Loss
↓
Backward
↓
Gradient calculated

```

But we haven't asked Adam to update anything.

In [0]:
print(
    "Weight before optimizer step:",
    model.layer1.weight[0, 0].item(),
)

It should still match weight_before.

This gives us a very important distinction:

loss.backward() CALCULATES gradients but DOES NOT update weights

##### 18. Update Parameters with Adam

In [0]:
optimizer.step()

Adam looks at the gradients calculated by: loss.backward()

and uses:

Current gradient
+
Running gradient statistics
+
Learning rate

to update the trainable parameters.

``` text

loss.backward()
      ↓
Gradients calculated
      ↓
optimizer.step()
      ↓
Adam reads gradients
      ↓
Adam calculates updates
      ↓
Weights + biases change

```

##### 19. Verify Weight Changed

In [0]:
weight_after = (
    model.layer1.weight[0, 0]
    .detach()
    .clone()
)

print(
    "Weight before:",
    weight_before.item(),
)

print(
    "Weight after:",
    weight_after.item(),
)

print(
    "Weight change:",
    (
        weight_after
        - weight_before
    ).item(),
)

##### One Complete Training Step

In [0]:
optimizer.zero_grad()

logits = model(X_batch)

loss = criterion(
    logits,
    y_batch,
)

loss.backward()

optimizer.step()

optimizer.zero_grad()
- clear old gradients

model(X_batch)
- forward propagation
- calculate logits

criterion(logits, y_batch)
- compare prediction with actual
- calculate loss

loss.backward()
- backpropagation
- calculate gradients

optimizer.step()
- Adam uses gradients
- update weights and biases

This is one training iteration / one batch update.

##### 18. Batch → Epoch → Training

ONE TRAINING STEP

``` text

32 customers
↓
one parameter update

```


155 TRAINING STEPS

``` text

≈ all 4,930 training customers
↓
ONE EPOCH

```

``` text 

20 EPOCHS
↓
training dataset seen approximately 20 times
↓
~3,100 optimizer updates

```

weight[0,0]
→ current parameter value

weight.grad[0,0]
→ how the loss is sensitive to changing that parameter

.item() converts a PyTorch tensor containing one value into a normal Python number.

``` text

tensor(-0.0397)
     ↓
.item()
     ↓
-0.03967764601111412

```

works when the tensor has exactly one element.

##### 20. Reset for the Real Training Run

In [0]:
#Create a fresh model:
model = TelcoChurnNN(
    input_size=input_size,
    hidden_size_1=HIDDEN_SIZE_1,
    hidden_size_2=HIDDEN_SIZE_2,
)

print(model)

In [0]:
#Create a fresh loss function:
criterion = nn.BCEWithLogitsLoss()

In [0]:
# Create a fresh Adam optimizer:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

So for a clean experiment:

Fresh Model
    +
Fresh Optimizer
    =
Fresh Training Run

##### 21. Initialize Training History and Checkpointing

In [0]:
#Before starting epochs, create two empty lists:

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_epoch = 0
best_model_state = None

epochs_without_improvement = 0

best_val_loss
→ best validation loss seen so far

best_epoch
→ epoch that produced it

best_model_state
→ saved copy of its weights and biases

epochs_without_improvement
→ early-stopping counter

##### 22. Full Training + Validation + Checkpointing + Early Stopping

In [0]:
for epoch in range(NUM_EPOCHS):

    # ---------------------------------------------------------
    # Training Phase
    # ---------------------------------------------------------

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
    )

    # ---------------------------------------------------------
    # Validation Phase
    # ---------------------------------------------------------

    val_loss = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
    )

    # ---------------------------------------------------------
    # Store Training History
    # ---------------------------------------------------------

    train_losses.append(
        train_loss
    )

    val_losses.append(
        val_loss
    )

    # ---------------------------------------------------------
    # Best-Model Checkpointing
    # ---------------------------------------------------------

    if val_loss < best_val_loss - MIN_DELTA:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    # ---------------------------------------------------------
    # Epoch Results
    # ---------------------------------------------------------

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    # ---------------------------------------------------------
    # Early Stopping
    # ---------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"Early stopping triggered at epoch {epoch + 1}."
        )

        break

##### 23. Restore Best Model

In [0]:
if best_model_state is None:
    raise RuntimeError(
        "Training completed without creating a best-model checkpoint."
    )

model.load_state_dict(
    best_model_state
)

model.state_dict()
→ GET / snapshot parameters

model.load_state_dict()
→ SET / restore parameters

model = best validation checkpoint

##### 24. Report Best Training Result

In [0]:
print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation loss:",
    round(best_val_loss, 4),
)

print(
    "Epochs actually run:",
    len(train_losses),
)

##### 25. Create Training Summary

In [0]:
training_summary = {
    "configured_epochs": NUM_EPOCHS,
    "epochs_completed": len(train_losses),
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "final_recorded_train_loss": train_losses[-1],
    "final_recorded_val_loss": val_losses[-1],
}

training_summary

final_recorded_val_loss
→ last executed epoch

best_validation_loss
→ best epoch

restored model
→ best epoch

##### 26. Plot Training vs Validation Loss

In [0]:
epochs_ran = range(
    1,
    len(train_losses) + 1,
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    epochs_ran,
    train_losses,
    label="Training Loss",
)

plt.plot(
    epochs_ran,
    val_losses,
    label="Validation Loss",
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Training vs Validation Loss"
)

plt.legend()

plt.grid(True)

plt.show()

##### 27. Optional Final Sanity Verification

In [0]:
X_check, y_check = next(
    iter(val_loader)
)

model.eval()

with torch.no_grad():

    check_logits = model(
        X_check
    )

print(
    "Validation batch:",
    X_check.shape,
)

print(
    "Model output:",
    check_logits.shape,
)

print(
    "Target shape:",
    y_check.shape,
)

because validation is asking:

How well does the model currently perform on data it did not train on?

We don't want validation customers teaching the model.


One complete epoch is therefore:

``` text

                EPOCH 1

             train_loader
                  ↓
            155 batches
                  ↓
        Forward → Loss
                  ↓
             Backward
                  ↓
                Adam
                  ↓
          Update parameters
                  ↓
        Average Train Loss
                  ↓
                  │
                  │
             val_loader
                  ↓
             33 batches
                  ↓
              Forward
                  ↓
                Loss
                  ↓
          NO Backpropagation
                  ↓
          NO Adam Update
                  ↓
       Average Validation Loss
                  ↓
                  │
                  ▼
      Train Loss + Validation Loss
                  ↓
                Store
                  ↓
               EPOCH 2

```

And repeat until Epoch 20.

state_dict()       = GET / take snapshot

load_state_dict()  = SET / restore snapshot

Checkpointing
→ saves best weights

Early stopping
→ stops unnecessary training

Validation loss
→ decides which model is best


``` text

Gold Delta Table
        ↓
Reusable Data Preparation
        ↓
train_loader / val_loader
        ↓
Fresh Neural Network
        ↓
BCEWithLogitsLoss
        ↓
Adam
        ↓

ONE-BATCH LEARNING VERIFICATION
        ↓
zero_grad()
        ↓
forward
        ↓
loss
        ↓
backward
        ↓
gradient
        ↓
optimizer.step()
        ↓
prove weight changed

        ↓

RESET FRESH MODEL

        ↓

FULL TRAINING

Epoch
 ├── Train All Batches
 │      ↓
 │   Forward
 │      ↓
 │    Loss
 │      ↓
 │   Backward
 │      ↓
 │    Adam
 │
 └── Validate
        ↓
     Forward Only
        ↓
   Validation Loss
        ↓
   Checkpoint Best Model
        ↓
   Early-Stopping Check
        ↓
       Next Epoch

        ↓

Restore Best Checkpoint
        ↓
Training Summary
        ↓
Loss Curve
        ↓
Ready for Test Evaluation

```


``` text 

Best model restored
        ↓
Start MLflow run
        ↓
Log training configuration
        ↓
Log best epoch + validation loss
        ↓
Log PyTorch model
        ↓
Log fitted preprocessor
        ↓
Save run_id
        ↓
Verify artifacts

```

##### 28. MLflow Experiment Setup

In [0]:
import mlflow
import mlflow.pytorch

In [0]:
MLFLOW_EXPERIMENT_NAME = (
    "/Users/sujathakrishna2811@gmail.com/telco_churn_neural_network_experiment"
)

mlflow.set_experiment(
    MLFLOW_EXPERIMENT_NAME
)


``` text

MLflow Experiment
│
├── Run 1
│    ├── Parameters
│    ├── Metrics
│    └── Artifacts
│
├── Run 2
│    ├── Parameters
│    ├── Metrics
│    └── Artifacts
│
└── Run 3

```

one complete neural-network training attempt = one MLflow run.

Experiment

``` text

telco_churn_neural_network_experiment
        │
        ├── Run A
        │    hidden1 = 32
        │    hidden2 = 16
        │    lr = 0.001
        │    batch = 32
        │    best_val_loss = 0.4180
        │
        └── Future Run B
             hidden1 = 64
             hidden2 = 32
             lr = 0.0005
             ...
```

##### 29 . What Should We Log?

There are three major categories:
- Parameters
- Metrics
- Artifacts

Parameters describe how we configured training:

- hidden_size_1
- hidden_size_2
- learning_rate
- batch_size
- max_epochs
- patience
- min_delta
- random_state

Metrics describe what happened:

- best_validation_loss
- best_epoch
- epochs_completed

Artifacts are actual objects/files:

- PyTorch model
- fitted preprocessor
- training-loss history

Parameter → What did I configure?

Metric → How well did it perform?

Artifact → What did the run produce?

##### 30. Complete MLflow Block

In [0]:
model.load_state_dict(
    best_model_state
)

In [0]:
import joblib
import mlflow
import mlflow.pytorch

from mlflow.models import infer_signature


# ---------------------------------------------------------
# Prepare Model Signature + Input Example
# ---------------------------------------------------------

model.eval()

input_example = (
    X_batch[:1]
    .detach()
    .cpu()
    .numpy()
)

with torch.no_grad():

    example_output = model(
        X_batch[:1]
    )

example_output = (
    example_output
    .detach()
    .cpu()
    .numpy()
)

signature = infer_signature(
    input_example,
    example_output,
)


# ---------------------------------------------------------
# Start MLflow Run
# ---------------------------------------------------------

with mlflow.start_run(
    run_name="telco_churn_nn_training"
) as run:

    run_id = run.info.run_id

    # -----------------------------------------------------
    # Log Parameters
    # -----------------------------------------------------

    mlflow.log_params(
        {
            "input_size": input_size,
            "hidden_size_1": HIDDEN_SIZE_1,
            "hidden_size_2": HIDDEN_SIZE_2,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "max_epochs": NUM_EPOCHS,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "random_state": RANDOM_STATE,
        }
    )

    # -----------------------------------------------------
    # Log Summary Metrics
    # -----------------------------------------------------

    mlflow.log_metrics(
        {
            "best_validation_loss": best_val_loss,
            "best_epoch": best_epoch,
            "epochs_completed": len(train_losses),
            "final_train_loss": train_losses[-1],
            "final_validation_loss": val_losses[-1],
        }
    )

    # -----------------------------------------------------
    # Log Per-Epoch Losses
    # -----------------------------------------------------

    for epoch_index, (
        train_loss,
        val_loss,
    ) in enumerate(
        zip(
            train_losses,
            val_losses,
        ),
        start=1,
    ):

        mlflow.log_metric(
            "train_loss",
            train_loss,
            step=epoch_index,
        )

        mlflow.log_metric(
            "validation_loss",
            val_loss,
            step=epoch_index,
        )

    # -----------------------------------------------------
    # Log Fitted Preprocessor
    # -----------------------------------------------------

    preprocessor_path = (
        "/tmp/telco_nn_preprocessor.joblib"
    )

    joblib.dump(
        preprocessor,
        preprocessor_path,
    )

    mlflow.log_artifact(
        preprocessor_path,
        artifact_path="preprocessing",
    )

    # -----------------------------------------------------
    # Log Best PyTorch Model
    # -----------------------------------------------------

    mlflow.pytorch.log_model(
        pytorch_model=model,
        name="model",
        input_example=input_example,
        signature=signature,
    )


print(
    "MLflow Run ID:",
    run_id,
)

##### 31. Verify What We Persisted

In [0]:
print(
    "MLflow Run ID:",
    run_id,
)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation loss:",
    round(
        best_val_loss,
        4,
    ),
)

print(
    "Input example shape:",
    input_example.shape,
)

print(
    "Example output shape:",
    example_output.shape,
)

##### MLflow Run Summary

The best restored neural-network model was persisted to MLflow together with:

- Training hyperparameters
- Best validation metrics
- Per-epoch training loss
- Per-epoch validation loss
- Fitted preprocessing pipeline
- PyTorch model artifact
- Input example
- Model signature

Latest successful MLflow Run ID: 6b15c39ebfb04e3a9c5803acf2f279a9

Best epoch: 8

Best validation loss: 0.4185

Input shape: (1, 45)

Output shape: (1, 1)

##### Key Learnings

1. Neural-network training happens batch by batch.

2. One training step performs:

   optimizer.zero_grad()
   → clears previous gradients

   model(X_batch)
   → performs forward propagation

   criterion(logits, y_batch)
   → calculates loss

   loss.backward()
   → calculates gradients using backpropagation

   optimizer.step()
   → updates weights and biases

3. One complete pass through all training batches is one epoch.

4. Training loss measures model fit on the training data.

5. Validation loss measures generalization to unseen validation data.

6. Validation data does not update model parameters.

7. model.train() enables training behavior.

8. model.eval() enables evaluation behavior.

9. torch.no_grad() disables unnecessary gradient tracking during validation and inference.

10. BCEWithLogitsLoss expects raw logits.

11. Sigmoid should not be manually applied before BCEWithLogitsLoss.

12. Adam uses gradients and running gradient statistics to update model parameters.

13. Learning rate controls the overall size of optimizer parameter updates.

14. Model checkpointing stores the parameters from the best validation epoch.

15. model.state_dict() retrieves the model parameter state.

16. model.load_state_dict() restores previously saved parameters.

17. Early stopping prevents unnecessary training when validation performance stops improving.

18. PATIENCE controls how many consecutive non-improving epochs are allowed.

19. MIN_DELTA defines the minimum improvement required to count as meaningful.

20. NUM_EPOCHS is the maximum number of epochs, not necessarily the number actually executed.

21. The fitted preprocessor must be preserved because the neural network expects the same transformed 45-feature representation during future inference.

22. MLflow provides a reproducible handoff between training and evaluation by storing the trained model, preprocessing artifacts, parameters, metrics, and model signature.

23. The test dataset remains untouched during training, validation, checkpoint selection, and early stopping.

##### Conclusion


This notebook trained the Telco Churn neural network using PyTorch.

The training workflow implemented:

``` text

Training Batch
      ↓
Forward Propagation
      ↓
Logits
      ↓
BCEWithLogitsLoss
      ↓
Loss
      ↓
Backpropagation
      ↓
Gradients
      ↓
Adam Optimizer
      ↓
Updated Weights + Biases
      ↓
Validation
      ↓
Best-Model Checkpointing
      ↓
Early Stopping
      ↓
Restore Best Model
      ↓
Persist Model + Preprocessor to MLflow

```


The model was trained using a maximum configured epoch count while validation loss determined the best checkpoint.

The best model was restored before persistence.

The trained PyTorch model, fitted preprocessing pipeline, model signature, input example, hyperparameters, and training metrics were successfully logged to MLflow.

The model is now ready for unbiased evaluation using the untouched test dataset.

##### Next Notebook

##### 05 - Evaluate Model

## Next Notebook

### 05_evaluate_model

The next notebook will load the trained model and preprocessing artifacts from MLflow and evaluate the model on the untouched test dataset.

The evaluation workflow will include:

``` text

MLflow Run
      ↓
Load Best PyTorch Model
      ↓
Load Fitted Preprocessor
      ↓
Prepare Test Data
      ↓
Forward Propagation
      ↓
Logits
      ↓
Sigmoid
      ↓
Churn Probabilities
      ↓
Classification Threshold
      ↓
Predicted Classes
      ↓
Evaluation Metrics

```

Metrics:

- Test Loss
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

The goal is to measure how well the selected neural network generalizes to customers that were not used during training or model selection.